<a href="https://colab.research.google.com/github/joseguzman1337/aws-financial-ai-agent/blob/main/invocation_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AWS Financial AI Agent - Invocation Demo

This notebook demonstrates end-to-end user authentication, live agent invocation via AWS Agentcore, and observability trace extraction.

### 0. Install Dependencies & Setup
Ensure the environment has the required libraries.

In [41]:
!pip install boto3 requests

import boto3
import requests
import json

access_token = None

print("Python environment ready!")

Python environment ready!


### 1. Authenticate with Amazon Cognito
Authentication is performed against the live Cognito User Pool provisioned via Terraform.

In [42]:
REGION = "us-east-1"
CLIENT_ID = "2r1ik1k110jbu6nfmuoegk5lns"
USERNAME = "analyst_user"
PASSWORD = "SecurePassword123!"

client = boto3.client('cognito-idp', region_name=REGION)
try:
    auth_response = client.initiate_auth(
        ClientId=CLIENT_ID,
        AuthFlow='USER_PASSWORD_AUTH',
        AuthParameters={'USERNAME': USERNAME, 'PASSWORD': PASSWORD}
    )
    access_token = auth_response['AuthenticationResult']['AccessToken']
    print(f"✅ Cognito Authentication Successful.")
except Exception as e:
    print(f"❌ Authentication Failed: {str(e)}")

✅ Cognito Authentication Successful.


### 2. Invoke the Agentcore Streaming Endpoint
This cell calls the live AWS Agentcore runtime. Responses are streamed in real-time.

In [46]:
AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:162187491349:runtime/Financial_Analyst_Agent-s5aas5HZhy"
ACCOUNT_ID = "162187491349"
# Note: AgentCore requires accountId as a query parameter for direct HTTPS invocations
AGENTCORE_URL = f"https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/{AGENT_ARN}/invocations?accountId={ACCOUNT_ID}"
SESSION_ID = "demo-session-recruiters-2026"

def query_financial_agent(prompt: str):
    if access_token is None:
        print("❌ ERROR: Access token is missing. Please run Cell 1 successfully first.")
        return

    print(f"\n--- Query: {prompt} ---")

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": SESSION_ID
    }

    try:
        response = requests.post(
            AGENTCORE_URL,
            headers=headers,
            json={"prompt": prompt},
            stream=True
        )

        if response.status_code != 200:
            print(f"❌ Error {response.status_code}: {response.text}")
            return

        for line in response.iter_lines():
            if line:
                decoded_line = line.decode('utf-8')
                if decoded_line.startswith("data: "):
                    data = json.loads(decoded_line[6:])
                    print(data.get("event", ""), end="", flush=True)
    except Exception as e:
        print(f"❌ Request Failed: {str(e)}")

print("✅ AWS Agent core configuration loaded.")

✅ Agent core configuration loaded.


### 3. Execute Required Financial Queries

In [44]:
queries = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
    "Compare Amazon's recent stock performance to what analysts predicted in their reports.",
    "I’m researching AMZN give me the current price and any relevant information about their AI business.",
    "What is the total amount of office space Amazon owned in North America in 2024?"
]

for q in queries:
    query_financial_agent(q)


--- Query: What is the stock price for Amazon right now? ---
❌ Error 404: <UnknownOperationException/>


--- Query: What were the stock prices for Amazon in Q4 last year? ---
❌ Error 404: <UnknownOperationException/>


--- Query: Compare Amazon's recent stock performance to what analysts predicted in their reports. ---
❌ Error 404: <UnknownOperationException/>


--- Query: I’m researching AMZN give me the current price and any relevant information about their AI business. ---
❌ Error 404: <UnknownOperationException/>


--- Query: What is the total amount of office space Amazon owned in North America in 2024? ---
❌ Error 404: <UnknownOperationException/>



### 4. Observability Verification
To verify the traces in Langfuse, the recruiter can use the public monitoring session ID below:

**Session ID:** `demo-session-recruiters-2026`